# Task 8 — Mask-based anomaly segmentation baselines with EoMT

This notebook evaluates EoMT on the anomaly segmentation validation datasets used in Task 7.

The pipeline:
1. mounts Google Drive and prepares the repository paths;
2. extracts the anomaly datasets locally in the Colab runtime;
3. evaluates the selected EoMT checkpoints with MSP, MaxLogit-like, Max Entropy and RbA at `T=1`;
4. saves the raw EoMT outputs locally and reuses them for temperature scaling;
5. exports report-ready CSV and Excel tables to Google Drive.

The intermediate mask/class logits are saved only in the local Colab runtime (`/content/saved_logits`) and are not persisted on Drive.


## 1. Environment setup, Drive and repository paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "MaskArchitectureAnomaly_CourseProject"
EOMT_ROOT = PROJECT_ROOT / "eomt"
EVAL_DIR = PROJECT_ROOT / "eval"
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"
RESULTS_ROOT = LARGE_FILES.parent / "results" / "task8"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

for p in (PROJECT_ROOT, EOMT_ROOT, EVAL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("PROJECT_ROOT:", PROJECT_ROOT, "| exists:", PROJECT_ROOT.exists())
print("EOMT_ROOT:", EOMT_ROOT, "| exists:", EOMT_ROOT.exists())
print("EVAL_DIR:", EVAL_DIR, "| exists:", EVAL_DIR.exists())
print("RESULTS_ROOT:", RESULTS_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EOMT_ROOT exists: True


## 2. Data, weights and output paths

In [ ]:
import zipfile
import glob

WEIGHTS_ROOT = LARGE_FILES / "weights"
ANOMALY_ZIP = LARGE_FILES / "datasets" / "anomaly" / "Anomaly_Validation_Datasets.zip"

# Datasets are extracted to the local Colab disk for faster image loading.
LOCAL_DATA_DIR = Path("/content/anomaly_data")
DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"

# Persistent result files on Drive.
RESULTS_CSV = RESULTS_ROOT / "eomt_last.csv"
EXCEL_OUTPUT_PATH = RESULTS_ROOT / "task8_results_last.xlsx"
TEMP_EXCEL_PATH = RESULTS_ROOT / "task8_temperature_results_last.xlsx"

# Temporary local folder for saved EoMT outputs. This folder is lost when the runtime is reset.
LOCAL_LOGITS_DIR = Path("/content/saved_logits")

print("ANOMALY_ZIP:", ANOMALY_ZIP, "| exists:", ANOMALY_ZIP.exists())
print("WEIGHTS_ROOT:", WEIGHTS_ROOT, "| exists:", WEIGHTS_ROOT.exists())
print("RESULTS_CSV:", RESULTS_CSV)
print("EXCEL_OUTPUT_PATH:", EXCEL_OUTPUT_PATH)
print("TEMP_EXCEL_PATH:", TEMP_EXCEL_PATH)
print("LOCAL_LOGITS_DIR:", LOCAL_LOGITS_DIR)


PROJECT_ROOT: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject
EOMT_ROOT: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt | exists: True
EVAL_DIR: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eval | exists: True
ANOMALY_ZIP: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/datasets/anomaly/Anomaly_Validation_Datasets.zip | exists: True
WEIGHTS_ROOT: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights | exists: True
RESULTS_CSV: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv


## 3. Extract the anomaly validation datasets

The anomaly zip is extracted to `/content` to avoid repeatedly reading images from Drive during inference.


In [ ]:
print("Zip exists:", ANOMALY_ZIP.exists())

if not LOCAL_DATA_DIR.exists():
    print("Extracting zip temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Done.")
else:
    print("Already extracted in this runtime.")

print("DATA_ROOT exists:", DATA_ROOT.exists())
if DATA_ROOT.exists():
    for p in sorted(DATA_ROOT.iterdir()):
        print("-", p.name)


Zip exists: True
Already extracted in this runtime.
DATA_ROOT: /content/anomaly_data/Validation_Dataset
DATA_ROOT exists: True
Contenuto DATA_ROOT:
- .DS_Store
- FS_LostFound_full
- RoadAnomaly
- RoadAnomaly21
- RoadObsticle21
- fs_static


## 4. Define anomaly datasets

In [4]:
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")


FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


## 5. Select EoMT checkpoints

The project requires the evaluation of three EoMT checkpoints:
- the COCO-trained panoptic checkpoint;
- the Cityscapes-trained semantic checkpoint;
- the head-only baseline COCO-to-Cityscapes checkpoint;
- the fine-tuned COCO-to-Cityscapes checkpoint.


In [ ]:
EOMT_COCO_WEIGHTS = WEIGHTS_ROOT / "eomt_coco.bin"
EOMT_CITYSCAPES_WEIGHTS = WEIGHTS_ROOT / "eomt_cityscapes.bin"
EOMT_FINETUNED_WEIGHTS = WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage1_head" / "stage1_weights.bin"
EOMT_BASELINE_WEIGHTS = WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_head_cache" / "head_only_weights_512.bin"


checkpoints = {
    "eomt_coco": {
        "weights": EOMT_COCO_WEIGHTS,
        "preset": "coco",
    },
     "eomt_cityscapes": {
         "weights": EOMT_CITYSCAPES_WEIGHTS,
         "preset": "cityscapes",
     },
     "eomt_baseline": {
         "weights": EOMT_BASELINE_WEIGHTS,
         "preset": "finetuned",
     },
     "eomt_finetuned": {
         "weights": EOMT_FINETUNED_WEIGHTS,
         "preset": "finetuned",
     },
}

for name, cfg in checkpoints.items():
    print(name, "->", cfg["weights"], "| exists:", cfg["weights"].exists())

eomt_baseline -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_head_cache/head_only_weights_512.bin | exists: True


## 6. Check checkpoint compatibility

This cell verifies that each checkpoint has the expected number of queries and classes for its preset.


In [ ]:
import torch

def inspect_ckpt(path):
    sd = torch.load(path, map_location="cpu")
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
    num_q = num_classes = None
    for k, v in sd.items():
        kk = k.replace("._orig_mod", "").replace("network.", "").replace("module.", "")
        if kk.endswith("q.weight"):
            num_q = v.shape[0]
        if kk.endswith("class_head.weight"):
            num_classes = v.shape[0] - 1
    return num_q, num_classes

PRESET_EXPECTED = {
    "coco": (200, 133),
    "cityscapes": (100, 19),
    "finetuned": (200, 19),
}

for name, cfg in checkpoints.items():
    if not cfg["weights"].exists():
        print(f"{name}: weights not found"); continue
    q, c = inspect_ckpt(cfg["weights"])
    exp_q, exp_c = PRESET_EXPECTED[cfg["preset"]]
    ok = (q == exp_q) and (c == exp_c)
    flag = "OK" if ok else "CHECK THIS PRESET"
    print(f"{name:24s} preset={cfg['preset']:10s} num_q={q} num_classes={c} "
          f"(expected {exp_q}/{exp_c}) -> {flag}")


eomt_baseline                preset=finetuned   num_q=200 num_classes=19 (atteso 200/19) OK


## 7. Baseline evaluation at temperature `T = 1`

For each checkpoint and dataset, the model forward pass is executed once per image. The raw EoMT outputs are saved locally as `.pt` files:
- `ml`: mask logits;
- `cl`: query class logits;
- `gt_v`: flattened valid anomaly ground truth;
- `valid`: valid-pixel mask;
- `pattern`: original dataset glob pattern.

The four Task 8 post-hoc methods are computed at `T=1`: MSP, MaxLogit-like, Max Entropy and RbA.

In [ ]:
import shutil
import time
from PIL import Image
import numpy as np
from types import SimpleNamespace
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
import evalAnomaly_eomt as E 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True
print("Device:", DEVICE)

# Start from a clean local logits directory and a clean CSV.
if LOCAL_LOGITS_DIR.exists():
    shutil.rmtree(LOCAL_LOGITS_DIR)
LOCAL_LOGITS_DIR.mkdir(parents=True, exist_ok=True)

if RESULTS_CSV.exists():
    RESULTS_CSV.unlink()

methods_baseline = ["msp", "maxlogit", "entropy", "rba"]
target_size = (E.IMG_HEIGHT, E.IMG_WIDTH)


def build_model(weights_path, preset):
    args = SimpleNamespace(
        preset=preset, 
        num_blocks=3, 
        patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
    )
    args = E.apply_preset(args)
    model = E.build_eomt(args)
    print(f"Loading weights {os.path.basename(str(weights_path))}")
    model = E.load_eomt_weights(model, str(weights_path))
    return model.to(DEVICE).eval(), args

for checkpoint_name, cfg in checkpoints.items():
    if not cfg["weights"].exists():
        print(f"Skip {checkpoint_name}: weights not found")
        continue
        
    model, args = build_model(cfg["weights"], cfg["preset"])

    for dataset_name, pattern in datasets.items():
        paths = sorted(glob.glob(str(pattern)))
        if not paths: continue

        scores_t1 = {m: [] for m in methods_baseline}
        gts_t1 = {m: [] for m in methods_baseline}
        input_pattern_str = str(pattern)

        print(f"\n[FORWARD] {checkpoint_name} -> {dataset_name} ({len(paths)} images)")
        t_ds = time.time()
        
        for idx, path in enumerate(paths):
            img = E.input_transform(Image.open(path).convert("RGB")).unsqueeze(0).float().to(DEVICE)
            
            with torch.no_grad():
                with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=(USE_AMP and DEVICE.type == "cuda")):
                    ml_layers, cl_layers = model(img)
                
                # Move outputs to CPU immediately to keep GPU memory low.
                ml = ml_layers[-1].float().cpu()
                cl = cl_layers[-1].float().cpu()
                
                # Load and align the anomaly ground truth.
                pathGT = E.get_gt_path(path)
                if not os.path.exists(pathGT): continue

                gt = E.convert_gt(np.array(E.target_transform(Image.open(pathGT))), pathGT)
                if 1 not in np.unique(gt): continue
                
                valid = (gt == 0) | (gt == 1)
                gt_v = gt[valid].astype(np.uint8)

                # Save the raw mask-architecture outputs.
                payload = {
                    "ml": ml, "cl": cl, "gt_v": gt_v, "valid": valid, "pattern": input_pattern_str
                }
                torch.save(payload, f"{LOCAL_LOGITS_DIR}/{checkpoint_name}_{dataset_name}_img_{idx}.pt")

                ml_d = ml.to(DEVICE)
                cl_d = cl.to(DEVICE)

                semantic_scores, semantic_probs = E.eomt_to_pixel_scores(
                    ml_d,
                    cl_d,
                    target_size,
                    temperature=1.0,
                )

                for m in methods_baseline:
                    if m == "rba":
                        s = E.compute_rba_score(
                            ml_d,
                            cl_d,
                            target_size,
                            temperature=1.0,
                        )
                    else:
                        s = E.compute_eomt_anomaly_score(
                            semantic_scores,
                            semantic_probs,
                            method=m,
                        )
                    
                    s_np = s.squeeze(0).float().cpu().numpy()[valid].astype(np.float32)
                    scores_t1[m].append(s_np)
                    gts_t1[m].append(gt_v)

        print(f"Dataset processed in {time.time() - t_ds:.1f}s")

        # Write T=1.0 results to the Drive CSV.
        for m in methods_baseline:
            if not gts_t1[m]: continue

            label = np.concatenate(gts_t1[m])
            out = np.concatenate(scores_t1[m])
            auprc = average_precision_score(label, out)
            fpr95 = fpr_at_95_tpr(out, label)

            args.checkpoint_name = checkpoint_name
            args.weights = str(cfg["weights"])
            args.input = [input_pattern_str]
            args.method = m
            args.temperature = 1.0
            args.results_csv = str(RESULTS_CSV)
            
            E.save_csv(args, auprc, fpr95, num_images=len(gts_t1[m]), num_pixels=len(label), num_anomaly_pixels=int(label.sum()))
            print(f"  ->  T=1.0 | {m:8s} -> AUPRC={auprc*100:.2f}  FPR95={fpr95*100:.2f}")

    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

print("\nBaseline evaluation completed. Local logits saved in:", LOCAL_LOGITS_DIR)

  [Modello] Caricamento pesi: head_only_weights_512.bin
Interpolating pos_embed from (1, 1024, 768) to (1, 2048, 768)
Checkpoint caricato: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_head_cache/head_only_weights_512.bin
Numero chiavi checkpoint: 197

[GPU RUN] eomt_baseline -> FS_LostFound_full (100 immagini)
  [dataset] Processato in 21.8s
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> BASELINE T=1.0 | msp      -> AUPRC=0.79  FPR95=67.41
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> BASELINE T=1.0 | maxlogit -> AUPRC=0.45  FPR95=93.31
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> BASELINE T=1.0 | entropy  -> AUPRC=1.08  FPR95=67.55
Risultati salvati in: /content/drive/MyDr

## 8. Temperature scaling from saved logits

This cell reuses the locally saved EoMT outputs and tests several temperatures without running the network again.

Temperature is applied to the query class logits before the softmax. For EoMT this can also affect the MaxLogit-like score, because the pixel-level semantic scores are reconstructed from temperature-scaled query probabilities.

In [ ]:
calib_temperatures = [0.5, 0.75, 1.1]
calib_methods = ["msp", "maxlogit", "entropy", "rba"]

target_size = (E.IMG_HEIGHT, E.IMG_WIDTH)

print("Starting offline temperature scaling from saved local logits...")

for checkpoint_name, cfg in checkpoints.items():
    if not cfg["weights"].exists(): continue
    
    # Create a dummy args object to reuse E.save_csv.
    from types import SimpleNamespace
    dummy_args = SimpleNamespace(
        preset=cfg["preset"], 
        num_blocks=3, 
        patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
        checkpoint_name=checkpoint_name, 
        weights=str(cfg["weights"]),
        results_csv=str(RESULTS_CSV)
    )
    dummy_args = E.apply_preset(dummy_args)

    for dataset_name in datasets.keys():
        saved_files = sorted(glob.glob(f"{LOCAL_LOGITS_DIR}/{checkpoint_name}_{dataset_name}_img_*.pt"))
        if not saved_files: continue

        print(f"\n[TEMPERATURE GRID] {checkpoint_name} -> {dataset_name} ({len(saved_files)} images)")
        t_start = time.time()
        
        # Initialize containers for dataset-level aggregation.
        all_gts = []
        # Create a dictionary to collect scores for each method-temperature pair.
        grid_scores = {(m, T): [] for m in calib_methods for T in calib_temperatures}
        input_pattern_str = ""

        # Read each saved file only once.
        for file_path in saved_files:
            # Load tensors and move them to the active device.
            payload = torch.load(file_path, map_location=DEVICE, weights_only=False) 

            # Post-processing can be done on GPU if available.
            ml = payload["ml"].to(DEVICE)
            cl = payload["cl"].to(DEVICE)
            gt_v = payload["gt_v"]
            valid = payload["valid"]
            input_pattern_str = payload["pattern"]
            
            all_gts.append(gt_v)

            with torch.no_grad():
                for T in calib_temperatures:
                    semantic_scores, semantic_probs = E.eomt_to_pixel_scores(
                        ml,
                        cl,
                        target_size,
                        temperature=T,
                    )

                    for m in calib_methods:
                        if m == "rba":
                            s = E.compute_rba_score(
                                ml,
                                cl,
                                target_size,
                                temperature=T,
                            )
                        else:
                            s = E.compute_eomt_anomaly_score(
                                semantic_scores,
                                semantic_probs,
                                method=m,
                            )
                        s_np = s.squeeze(0).float().cpu().numpy()[valid].astype(np.float32)
                        grid_scores[(m, T)].append(s_np)

        # Aggregate metrics over the full dataset.
        if all_gts:
            flat_gts = np.concatenate(all_gts)
            
            for m in calib_methods:
                for T in calib_temperatures:
                    flat_scores = np.concatenate(grid_scores[(m, T)])

                    auprc = average_precision_score(flat_gts, flat_scores)
                    fpr95 = fpr_at_95_tpr(flat_scores, flat_gts)

                    dummy_args.method = m
                    dummy_args.temperature = T
                    dummy_args.input = [input_pattern_str]
                    
                    E.save_csv(dummy_args, auprc, fpr95, num_images=len(all_gts), num_pixels=len(flat_gts), num_anomaly_pixels=int(flat_gts.sum()))
                    print(f"  -> method: {m:8s} | T = {T:<4} -> AUPRC: {auprc*100:.2f}% | FPR95: {fpr95*100:.2f}%")
            
            print(f"  Dataset completed in {time.time() - t_start:.1f}s")

print("\nTemperature scaling completed.")

--- INIZIO FASE 2: Temperature Scaling OTTIMIZZATO (Device: cuda) ---

[OFFLINE GRID SEARCH] eomt_baseline -> FS_LostFound_full (99 immagini)
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> Metodo: msp      | T = 0.5  -> AUPRC: 0.69% | FPR95: 69.51%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> Metodo: msp      | T = 0.75 -> AUPRC: 0.69% | FPR95: 69.17%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> Metodo: msp      | T = 1.1  -> AUPRC: 0.86% | FPR95: 65.78%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/eomt_baseline_only.csv
  -> Metodo: maxlogit | T = 0.5  -> AUPRC: 0.58% | FPR95: 92.50%
Risultati salvati in: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/e

## 9. Load and inspect the result CSV

In [ ]:
import pandas as pd

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    display(df.tail(30))
else:
    print("CSV not found:", RESULTS_CSV)


,checkpoint_name,preset,weights,input,method,temperature,num_classes,num_q,num_blocks,img_height,img_width,AUPRC,FPR95,num_images,num_pixels,num_anomaly_pixels
50,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,0.50,19,200,3,512,1024,27.995886,55.116351,60,31457280,3098336
51,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,0.75,19,200,3,512,1024,42.758979,47.633226,60,31457280,3098336
52,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,entropy,1.10,19,200,3,512,1024,58.839069,42.506768,60,31457280,3098336
53,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,0.50,19,200,3,512,1024,38.650490,87.193743,60,31457280,3098336
54,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,0.75,19,200,3,512,1024,44.550548,94.635252,60,31457280,3098336
55,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,rba,1.10,19,200,3,512,1024,26.963045,98.650987,60,31457280,3098336
56,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,0.50,19,200,3,512,1024,46.700927,26.948016,10,5060302,749109
57,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,0.75,19,200,3,512,1024,50.565130,18.978784,10,5060302,749109
58,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,msp,1.10,19,200,3,512,1024,55.196917,12.652020,10,5060302,749109
59,eomt_baseline,finetuned,/content/drive/MyDrive/FAIML_project_and_prese...,/content/anomaly_data/Validation_Dataset/RoadA...,maxlogit,0.50,19,200,3,512,1024,39.153716,52.345001,10,5060302,749109


## 10. Pivot tables for quick inspection

In [ ]:
if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in datasets.keys():
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    pivot_auprc = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="AUPRC",
        aggfunc="last",
    )

    pivot_fpr95 = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="FPR95",
        aggfunc="last",
    )

    print("AUPRC")
    display(pivot_auprc)

    print("FPR95")
    display(pivot_fpr95)
else:
    print("CSV not created yet.")

AUPRC


dataset                                         FS_LostFound_full  \
checkpoint_name preset    method   temperature                      
eomt_baseline   finetuned entropy  0.50                  0.695499   
                                   0.75                  0.816300   
                                   1.00                  1.083920   
                                   1.10                  1.215019   
                          maxlogit 0.50                  0.578255   
                                   0.75                  0.486958   
                                   1.00                  0.451292   
                                   1.10                  0.439863   
                          msp      0.50                  0.689997   
                                   0.75                  0.686555   
                                   1.00                  0.792375   
                                   1.10                  0.856998   
                          rba      0.50                  0.335604   
                                   0.75                  0.243905   
                                   1.00                  0.192022   
                                   1.10                  0.181004   

dataset                                         RoadAnomaly  RoadObsticle21  \
checkpoint_name preset    method   temperature                                
eomt_baseline   finetuned entropy  0.50           50.780867       53.635181   
                                   0.75           55.215121       77.774580   
                                   1.00           56.204331       91.761263   
                                   1.10           56.087736       92.755076   
                          maxlogit 0.50           39.153716       85.626235   
                                   0.75           41.023343       85.649620   
                                   1.00           45.729465       84.362942   
                                   1.10           48.896186       83.860252   
                          msp      0.50           46.700927       39.162291   
                                   0.75           50.565130       62.975557   
                                   1.00           53.816325       79.723836   
                                   1.10           55.196917       84.282501   
                          rba      0.50           16.756990       82.260338   
                                   0.75           11.962853       74.864967   
                                   1.00           11.771134       65.314751   
                                   1.10           12.218783       61.036310   

dataset                                         fs_static  
checkpoint_name preset    method   temperature             
eomt_baseline   finetuned entropy  0.50         15.372319  
                                   0.75         23.979981  
                                   1.00         35.243613  
                                   1.10         40.426748  
                          maxlogit 0.50         37.017052  
                                   0.75         38.897748  
                                   1.00         38.889617  
                                   1.10         38.702387  
                          msp      0.50         14.831413  
                                   0.75         21.630554  
                                   1.00         23.813234  
                                   1.10         25.030260  
                          rba      0.50         33.444198  
                                   0.75         28.833548  
                                   1.00         19.168483  
                                   1.10         16.368535

FPR95


dataset                                         FS_LostFound_full  \
checkpoint_name preset    method   temperature                      
eomt_baseline   finetuned entropy  0.50                 70.457954   
                                   0.75                 69.571923   
                                   1.00                 67.546633   
                                   1.10                 65.443744   
                          maxlogit 0.50                 92.501516   
                                   0.75                 92.498969   
                                   1.00                 93.313590   
                                   1.10                 93.388182   
                          msp      0.50                 69.508978   
                                   0.75                 69.174731   
                                   1.00                 67.414244   
                                   1.10                 65.778963   
                          rba      0.50                 98.269476   
                                   0.75                 98.024216   
                                   1.00                 98.120816   
                                   1.10                 98.302439   

dataset                                         RoadAnomaly  RoadObsticle21  \
checkpoint_name preset    method   temperature                                
eomt_baseline   finetuned entropy  0.50           22.703785       23.815689   
                                   0.75           18.079845        9.336599   
                                   1.00           31.346451        1.262251   
                                   1.10           35.192240        0.706219   
                          maxlogit 0.50           52.345001       34.277392   
                                   0.75           50.296635       32.848187   
                                   1.00           48.246483       33.589452   
                                   1.10           47.967906       45.276232   
                          msp      0.50           26.948016       26.182154   
                                   0.75           18.978784       14.377882   
                                   1.00           13.777764        5.670431   
                                   1.10           12.652020        3.321371   
                          rba      0.50           91.262419       86.525300   
                                   0.75           91.604134       96.519003   
                                   1.00           93.605923       99.966650   
                                   1.10           93.913891       99.996833   

dataset                                         fs_static  
checkpoint_name preset    method   temperature             
eomt_baseline   finetuned entropy  0.50         52.808225  
                                   0.75         32.865375  
                                   1.00         17.450993  
                                   1.10         13.642501  
                          maxlogit 0.50         15.772243  
                                   0.75         16.208906  
                                   1.00         18.565913  
                                   1.10         20.283772  
                          msp      0.50         58.100689  
                                   0.75         36.495432  
                                   1.00         29.486340  
                                   1.10         27.619314  
                          rba      0.50         77.314896  
                                   0.75         74.669778  
                                   1.00         79.718165  
                                   1.10         83.102349

## 11. Export the `T=1` baseline table to Excel

The mIoU column is optional. Fill the `MIOU_BY_CHECKPOINT` dictionary with the values obtained in Tasks 4/5 if you want them to appear in the final table.


In [12]:
!pip install xlsxwriter

In [ ]:
import os

DATASET_COLS = {
    "RoadAnomaly21": "SMIYC RA-21",
    "RoadObsticle21": "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static": "FS Static",
    "RoadAnomaly": "Road Anomaly",
}
MODEL_LABELS = {
    "eomt_coco": "EoMT COCO",
    "eomt_cityscapes": "EoMT Cityscapes",
    "eomt_finetuned": "EoMT finetuned",
    "eomt_baseline": "EoMT baseline"
}
METHOD_LABELS = {"msp": "MSP", "maxlogit": "MaxLogit", "entropy": "Max Entropy", "rba": "RbA"}

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    search_keys = sorted(DATASET_COLS.keys(), key=len, reverse=True)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in search_keys:
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)
    df = df[df["temperature"] == 1.0]

    col_tuples = [(lbl, metric) for lbl in DATASET_COLS.values() for metric in ("AuPRC", "FPR95")]
    data = {ct: [] for ct in col_tuples}
    index_tuples = []

    ckpt_list = list(checkpoints.keys()) if ('checkpoints' in locals() or 'checkpoints' in globals()) else list(MODEL_LABELS.keys())

    for ckpt in ckpt_list:            
        for m in METHOD_LABELS:              
            sub = df[(df["checkpoint_name"] == ckpt) & (df["method"] == m)]
            if sub.empty:
                continue
            index_tuples.append((MODEL_LABELS.get(ckpt, ckpt), METHOD_LABELS[m]))
            for ds_raw, ds_label in DATASET_COLS.items():
                r = sub[sub["dataset"] == ds_raw]
                a = round(float(r["AUPRC"].iloc[-1]), 2) if not r.empty else None
                f = round(float(r["FPR95"].iloc[-1]), 2) if not r.empty else None
                data[(ds_label, "AuPRC")].append(a)
                data[(ds_label, "FPR95")].append(f)

    table = pd.DataFrame(
        data,
        index=pd.MultiIndex.from_tuples(index_tuples, names=["Model", "Method"]),
    )
    table.columns = pd.MultiIndex.from_tuples(table.columns)

    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="xlsxwriter") as writer:
        table.to_excel(writer, sheet_name="Task8")

    print("Saved Excel table:", EXCEL_OUTPUT_PATH)
    display(table)
else:
    print("CSV not found. Run the evaluation first.")

Excel salvato: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/task8_results_baseline_only.xlsx


SMIYC RA-21        SMIYC RO-21        FS L&F         \
                                AuPRC  FPR95       AuPRC  FPR95  AuPRC  FPR95   
Model         Method                                                            
EoMT baseline MSP               53.82  13.78       79.72   5.67   0.79  67.41   
              MaxLogit          45.73  48.25       84.36  33.59   0.45  93.31   
              Max Entropy       56.20  31.35       91.76   1.26   1.08  67.55   
              RbA               11.77  93.61       65.31  99.97   0.19  98.12   

                          FS Static        Road Anomaly         
                              AuPRC  FPR95        AuPRC  FPR95  
Model         Method                                            
EoMT baseline MSP             23.81  29.49        41.44  47.26  
              MaxLogit        38.89  18.57        55.34  54.78  
              Max Entropy     35.24  17.45        54.96  44.68  
              RbA             19.17  79.72        31.94  98.04

## 12. Export temperature-scaling tables to Excel

For each checkpoint, this cell creates a sheet containing the results at each tested temperature plus a summary row. The `best t` row reports the best value independently for each metric, so it should be interpreted as the best achievable value across the tested temperatures, not necessarily as a single shared temperature for AuPRC and FPR95.

In [ ]:
# Export temperature-scaling results to Excel.

import pandas as pd
import os

TEMP_METHOD_LABELS = {"msp": "MSP", "entropy": "Max Entropy", "rba": "RbA", "maxlogit" : "Max Logit" }

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    search_keys = sorted(DATASET_COLS.keys(), key=len, reverse=True)

    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in search_keys:
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    col_tuples = [("", "mIoU")] + [(lbl, met) for lbl in DATASET_COLS.values()
                                   for met in ("AuPRC", "FPR95")]

    def vals_at_temp(g, T, ds_raw):
        r = g[(g["dataset"] == ds_raw) & (g["temperature"] == T)]
        if r.empty:
            return None, None
        return round(float(r["AUPRC"].iloc[-1]), 2), round(float(r["FPR95"].iloc[-1]), 2)

    def vals_best(g, ds_raw): 
        r = g[g["dataset"] == ds_raw]
        if r.empty:
            return None, None
        return round(float(r["AUPRC"].max()), 2), round(float(r["FPR95"].min()), 2)

    sheets = {}
    
    ckpt_list = list(checkpoints.keys()) if ('checkpoints' in locals() or 'checkpoints' in globals()) else ["eomt_coco", "eomt_cityscapes", "eomt_finetuned"]

    for ckpt in ckpt_list:
        data = {ct: [] for ct in col_tuples}
        index_labels = []
        for m, mlabel in TEMP_METHOD_LABELS.items():
            g = df[(df["checkpoint_name"] == ckpt) & (df["method"] == m)]
            if g.empty or g["temperature"].nunique() <= 1:
                continue
            temps = sorted(g["temperature"].unique())
            rows = []
            if 1.0 in temps:
                rows.append((mlabel, lambda ds, g=g: vals_at_temp(g, 1.0, ds)))
            for T in [t for t in temps if t != 1.0]:
                rows.append((f"{mlabel}(t={T})", lambda ds, g=g, T=T: vals_at_temp(g, T, ds)))
            rows.append((f"{mlabel} (best t)", lambda ds, g=g: vals_best(g, ds)))
            for label, getter in rows:
                index_labels.append(label)
                data[("", "mIoU")].append("----")
                for ds_raw, ds_label in DATASET_COLS.items():
                    a, f = getter(ds_raw)
                    data[(ds_label, "AuPRC")].append(a)
                    data[(ds_label, "FPR95")].append(f)
        if index_labels:
            table = pd.DataFrame(data, index=pd.Index(index_labels, name="Method"))
            table.columns = pd.MultiIndex.from_tuples(table.columns)
            sheets[ckpt] = table

    if not sheets:
        print("No temperature-scaling results found. Run the temperature-scaling cell first.")
    else:
        with pd.ExcelWriter(TEMP_EXCEL_PATH, engine="xlsxwriter") as writer:
            for ckpt, table in sheets.items():
                table.to_excel(writer, sheet_name=ckpt[:31])
                print("Saved temperature-scaling Excel table:", TEMP_EXCEL_PATH)
        print("Sheets:", list(sheets.keys()))
        display(list(sheets.values())[-1])
else:
    print("CSV not found. Run the evaluation first.")

Excel temperature salvato: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task8/task8_temperature_results_baseline_only.xlsx
Fogli (un checkpoint ciascuno): ['eomt_baseline']


SMIYC RA-21        SMIYC RO-21         FS L&F  \
                      mIoU       AuPRC  FPR95       AuPRC   FPR95  AuPRC   
Method                                                                     
MSP                   ----       53.82  13.78       79.72    5.67   0.79   
MSP(t=0.5)            ----       46.70  26.95       39.16   26.18   0.69   
MSP(t=0.75)           ----       50.57  18.98       62.98   14.38   0.69   
MSP(t=1.1)            ----       55.20  12.65       84.28    3.32   0.86   
MSP (best t)          ----       55.20  12.65       84.28    3.32   0.86   
Max Entropy           ----       56.20  31.35       91.76    1.26   1.08   
Max Entropy(t=0.5)    ----       50.78  22.70       53.64   23.82   0.70   
Max Entropy(t=0.75)   ----       55.22  18.08       77.77    9.34   0.82   
Max Entropy(t=1.1)    ----       56.09  35.19       92.76    0.71   1.22   
Max Entropy (best t)  ----       56.20  18.08       92.76    0.71   1.22   
RbA                   ----       11.77  93.61       65.31   99.97   0.19   
RbA(t=0.5)            ----       16.76  91.26       82.26   86.53   0.34   
RbA(t=0.75)           ----       11.96  91.60       74.86   96.52   0.24   
RbA(t=1.1)            ----       12.22  93.91       61.04  100.00   0.18   
RbA (best t)          ----       16.76  91.26       82.26   86.53   0.34   
Max Logit             ----       45.73  48.25       84.36   33.59   0.45   
Max Logit(t=0.5)      ----       39.15  52.35       85.63   34.28   0.58   
Max Logit(t=0.75)     ----       41.02  50.30       85.65   32.85   0.49   
Max Logit(t=1.1)      ----       48.90  47.97       83.86   45.28   0.44   
Max Logit (best t)    ----       48.90  47.97       85.65   32.85   0.58   

                            FS Static        Road Anomaly         
                      FPR95     AuPRC  FPR95        AuPRC  FPR95  
Method                                                            
MSP                   67.41     23.81  29.49        41.44  47.26  
MSP(t=0.5)            69.51     14.83  58.10        22.09  57.77  
MSP(t=0.75)           69.17     21.63  36.50        29.72  50.88  
MSP(t=1.1)            65.78     25.03  27.62        45.31  46.27  
MSP (best t)          65.78     25.03  27.62        45.31  46.27  
Max Entropy           67.55     35.24  17.45        54.96  44.68  
Max Entropy(t=0.5)    70.46     15.37  52.81        28.00  55.12  
Max Entropy(t=0.75)   69.57     23.98  32.87        42.76  47.63  
Max Entropy(t=1.1)    65.44     40.43  13.64        58.84  42.51  
Max Entropy (best t)  65.44     40.43  13.64        58.84  42.51  
RbA                   98.12     19.17  79.72        31.94  98.04  
RbA(t=0.5)            98.27     33.44  77.31        38.65  87.19  
RbA(t=0.75)           98.02     28.83  74.67        44.55  94.64  
RbA(t=1.1)            98.30     16.37  83.10        26.96  98.65  
RbA (best t)          98.02     33.44  74.67        44.55  87.19  
Max Logit             93.31     38.89  18.57        55.34  54.78  
Max Logit(t=0.5)      92.50     37.02  15.77        43.70  57.24  
Max Logit(t=0.75)     92.50     38.90  16.21        50.03  55.88  
Max Logit(t=1.1)      93.39     38.70  20.28        56.85  55.01  
Max Logit (best t)    92.50     38.90  15.77        56.85  54.78